# AuraGateway CUDA 12.9 Python startup inspection v1

Bounded diagnostic inspection of Kaggle target-Python startup customization. No package installation, model loading, requests, or qualification.


In [ ]:

from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

NOTEBOOK_NAME = "auragateway-cu129-python-startup-inspect-v1"
OUTPUT_DIRECTORY_NAME = "auragateway_cu129_python_startup_inspection_evidence_v1"
OUTPUT_ZIP = Path(f"/kaggle/working/{OUTPUT_DIRECTORY_NAME}.zip")
EVIDENCE_ROOT = Path(f"/kaggle/working/{OUTPUT_DIRECTORY_NAME}")
VENV_ROOT = Path("/kaggle/working/auragateway_cu129_startup_inspection_venv_v1")
MAX_EXCERPT = 30000

SOURCE_VERIFIER = "auragateway-cu129-offline-verifier-v5"
SOURCE_VERIFIER_NOTEBOOK_SHA256 = (
    "ad4d7a533de09965f5562db58c85cb785f2a7f8550906631142e26a34b6e59ab"
)
SOURCE_EVIDENCE_ZIP_SHA256 = (
    "303879f21a0245f566a6df39e950afe90e8f15799a819e889a3a75b20fc97ae6"
)
SOURCE_EXECUTION_LOG_SHA256 = (
    "1ff315f4438fa62bc3f2ad92a369b1f5fa3d4d836f27f2e4e209fd47b4cb2056"
)
SOURCE_FIRST_DIVERGENCE = "target_runtime_identity_before_install"
SOURCE_FAILURE_CLASS = "TARGET_PYTHON_STARTUP_CUSTOMIZATION_LEAK"

PROBE_SCRIPT = r"""
import importlib.util
import json
import os
import site
import sys
import sysconfig
from pathlib import Path

expected = Path(sys.argv[1]).resolve()
prefix = Path(sys.prefix).resolve()
base_prefix = Path(sys.base_prefix).resolve()
paths = sysconfig.get_paths()
target_site = (
    expected
    / "lib"
    / f"python{sys.version_info.major}.{sys.version_info.minor}"
    / "site-packages"
).resolve()

def module_origin(name: str) -> str | None:
    module = sys.modules.get(name)
    if module is None:
        return None
    value = getattr(module, "__file__", None)
    return None if value is None else str(value)

def is_external_package_path(value: str) -> bool:
    if not value:
        return False
    path = Path(value).resolve()
    if path == target_site or target_site in path.parents:
        return False
    return any(part in {"site-packages", "dist-packages"} for part in path.parts)

print(
    json.dumps(
        {
            "python": ".".join(str(item) for item in sys.version_info[:3]),
            "executable": str(Path(sys.executable).resolve()),
            "prefix": str(prefix),
            "base_prefix": str(base_prefix),
            "prefix_matches_expected": prefix == expected,
            "base_prefix_differs": base_prefix != prefix,
            "pythonpath_present": "PYTHONPATH" in os.environ,
            "pythonhome_present": "PYTHONHOME" in os.environ,
            "python_no_user_site": os.environ.get("PYTHONNOUSERSITE"),
            "user_site_enabled": site.ENABLE_USER_SITE,
            "pip_present": importlib.util.find_spec("pip") is not None,
            "target_site_packages": str(target_site),
            "target_site_packages_present": any(
                Path(item).resolve() == target_site
                for item in sys.path
                if item
            ),
            "external_package_paths": [
                item for item in sys.path if is_external_package_path(item)
            ],
            "sys_path": sys.path,
            "sitecustomize_loaded": "sitecustomize" in sys.modules,
            "sitecustomize_origin": module_origin("sitecustomize"),
            "usercustomize_loaded": "usercustomize" in sys.modules,
            "usercustomize_origin": module_origin("usercustomize"),
            "no_site_flag": sys.flags.no_site,
            "isolated_flag": sys.flags.isolated,
        },
        sort_keys=True,
    )
)
"""

CONTROLLED_BOOTSTRAP = r"""
import json
import site
import sys
import types
from pathlib import Path

expected = Path(sys.argv[1]).resolve()
payload = sys.argv[2]

def sentinel(name: str) -> types.ModuleType:
    module = types.ModuleType(name)
    module.__file__ = f"<auragateway-suppressed-{name}>"
    return module

sys.modules["sitecustomize"] = sentinel("sitecustomize")
sys.modules["usercustomize"] = sentinel("usercustomize")
site.main()

target_site = (
    expected
    / "lib"
    / f"python{sys.version_info.major}.{sys.version_info.minor}"
    / "site-packages"
).resolve()

cleaned = []
for value in sys.path:
    if not value:
        cleaned.append(value)
        continue
    path = Path(value).resolve()
    is_target = path == target_site or target_site in path.parents
    is_package_path = any(
        part in {"site-packages", "dist-packages"}
        for part in path.parts
    )
    if is_package_path and not is_target:
        continue
    cleaned.append(value)

if str(target_site) not in cleaned:
    cleaned.insert(0, str(target_site))

sys.path[:] = cleaned
exec(compile(payload, "<auragateway-target-probe>", "exec"), {"__name__": "__main__"})
"""


def canonical_json(payload: object) -> str:
    return json.dumps(
        payload,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    )


def streaming_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sanitize(text: str | bytes | None) -> str:
    if text is None:
        return ""
    decoded = (
        text.decode("utf-8", errors="replace")
        if isinstance(text, bytes)
        else text
    )
    replacements = {
        "/kaggle/input": "<input>",
        "/kaggle/working": "<working>",
        os.environ.get("HOME", ""): "<home>",
    }
    for source, replacement in replacements.items():
        if source:
            decoded = decoded.replace(source, replacement)
    return decoded[-MAX_EXCERPT:]


def offline_environment() -> dict[str, str]:
    environment = {**os.environ}
    environment.pop("PYTHONPATH", None)
    environment.pop("PYTHONHOME", None)
    environment["PYTHONNOUSERSITE"] = "1"
    environment["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
    environment["PIP_NO_INDEX"] = "1"
    environment["HF_HUB_OFFLINE"] = "1"
    environment["TRANSFORMERS_OFFLINE"] = "1"
    environment["VIRTUAL_ENV"] = str(VENV_ROOT)
    environment["PATH"] = os.pathsep.join(
        [
            str(VENV_ROOT / "bin"),
            *[
                item
                for item in environment.get("PATH", "").split(os.pathsep)
                if item and item != str(VENV_ROOT / "bin")
            ],
        ]
    )
    return environment


def run_command(
    role: str,
    argv: list[str],
    *,
    timeout: float,
    environment: dict[str, str] | None = None,
) -> dict[str, object]:
    started = time.monotonic()
    started_at = datetime.now(UTC).isoformat(timespec="seconds")
    env = offline_environment() if environment is None else environment
    try:
        result = subprocess.run(
            argv,
            check=False,
            capture_output=True,
            text=True,
            timeout=timeout,
            env=env,
        )
        return {
            "schema_version": "1.0.0",
            "command_role": role,
            "argv": [sanitize(item) for item in argv],
            "started_at": started_at,
            "duration_ms": int((time.monotonic() - started) * 1000),
            "returncode": result.returncode,
            "timed_out": False,
            "status": "PASSED" if result.returncode == 0 else "FAILED",
            "stdout_excerpt": sanitize(result.stdout),
            "stderr_excerpt": sanitize(result.stderr),
        }
    except subprocess.TimeoutExpired as exc:
        return {
            "schema_version": "1.0.0",
            "command_role": role,
            "argv": [sanitize(item) for item in argv],
            "started_at": started_at,
            "duration_ms": int((time.monotonic() - started) * 1000),
            "returncode": None,
            "timed_out": True,
            "status": "FAILED",
            "stdout_excerpt": sanitize(exc.stdout),
            "stderr_excerpt": sanitize(exc.stderr),
        }


def parse_stdout(record: dict[str, object]) -> dict[str, Any] | None:
    try:
        payload = json.loads(str(record["stdout_excerpt"]).strip())
    except json.JSONDecodeError:
        return None
    return payload if isinstance(payload, dict) else None


def write_record(name: str, payload: object) -> None:
    (EVIDENCE_ROOT / name).write_text(
        canonical_json(payload) + "\n",
        encoding="utf-8",
    )


def controlled_bootstrap_passed(record: dict[str, object]) -> bool:
    if record.get("status") != "PASSED":
        return False
    if "sitecustomize" in str(record.get("stderr_excerpt", "")).lower():
        return False
    payload = parse_stdout(record)
    if payload is None:
        return False
    expected_sitecustomize = "<auragateway-suppressed-sitecustomize>"
    expected_usercustomize = "<auragateway-suppressed-usercustomize>"
    return (
        payload.get("python") == "3.12.13"
        and payload.get("prefix_matches_expected") is True
        and payload.get("base_prefix_differs") is True
        and payload.get("pythonpath_present") is False
        and payload.get("pythonhome_present") is False
        and payload.get("python_no_user_site") == "1"
        and payload.get("user_site_enabled") is False
        and payload.get("pip_present") is False
        and payload.get("target_site_packages_present") is True
        and payload.get("external_package_paths") == []
        and payload.get("sitecustomize_loaded") is True
        and payload.get("sitecustomize_origin") == expected_sitecustomize
        and payload.get("usercustomize_loaded") is True
        and payload.get("usercustomize_origin") == expected_usercustomize
        and payload.get("no_site_flag") == 1
    )


def main() -> int:
    if os.environ.get("AURAGATEWAY_CUSTOMER_DATA_PRESENT") == "1":
        raise RuntimeError("customer data is prohibited")

    for path in (EVIDENCE_ROOT, VENV_ROOT):
        if path.exists():
            shutil.rmtree(path)
    if OUTPUT_ZIP.exists():
        OUTPUT_ZIP.unlink()
    EVIDENCE_ROOT.mkdir(parents=True)

    records: list[dict[str, object]] = []

    base_python = run_command(
        "base_python_runtime",
        [
            sys.executable,
            "-c",
            "import platform; print(platform.python_version())",
        ],
        timeout=30.0,
    )
    records.append(base_python)

    base_venv = run_command(
        "base_venv_import",
        [sys.executable, "-c", "import venv; print('ok')"],
        timeout=30.0,
    )
    records.append(base_venv)

    topology = run_command(
        "gpu_topology",
        [
            "nvidia-smi",
            "--query-gpu=index,name,memory.total,compute_cap,driver_version",
            "--format=csv,noheader,nounits",
        ],
        timeout=30.0,
    )
    records.append(topology)

    venv_create = run_command(
        "venv_create_without_pip",
        [
            sys.executable,
            "-m",
            "venv",
            "--without-pip",
            str(VENV_ROOT),
        ],
        timeout=180.0,
    )
    records.append(venv_create)

    target_python = VENV_ROOT / "bin" / "python"
    environment = offline_environment()

    if venv_create["status"] == "PASSED":
        modes = (
            (
                "sanitized_default_startup",
                [
                    str(target_python),
                    "-c",
                    PROBE_SCRIPT,
                    str(VENV_ROOT),
                ],
            ),
            (
                "isolated_mode_startup",
                [
                    str(target_python),
                    "-I",
                    "-c",
                    PROBE_SCRIPT,
                    str(VENV_ROOT),
                ],
            ),
            (
                "no_site_startup",
                [
                    str(target_python),
                    "-S",
                    "-c",
                    PROBE_SCRIPT,
                    str(VENV_ROOT),
                ],
            ),
            (
                "controlled_site_bootstrap",
                [
                    str(target_python),
                    "-S",
                    "-c",
                    CONTROLLED_BOOTSTRAP,
                    str(VENV_ROOT),
                    PROBE_SCRIPT,
                ],
            ),
        )
        for role, argv in modes:
            records.append(
                run_command(
                    role,
                    argv,
                    timeout=60.0,
                    environment=environment,
                )
            )

    observed = {
        str(record["command_role"]): record
        for record in records
    }
    controlled = observed.get("controlled_site_bootstrap")
    controlled_confirmed = (
        controlled is not None
        and controlled_bootstrap_passed(controlled)
    )

    if controlled_confirmed:
        disposition = "CONTROLLED_SITE_BOOTSTRAP_CONFIRMED"
        next_gate = "build_cu129_offline_runtime_compatibility_verifier_v6"
    else:
        disposition = "NO_SAFE_BOOTSTRAP_IDENTIFIED"
        next_gate = "inspect_python_startup_path_injection"

    summary = {
        "schema_version": "1.0.0",
        "diagnostic_id": NOTEBOOK_NAME,
        "captured_at": datetime.now(UTC).isoformat(timespec="seconds"),
        "inspection_status": "COMPLETED",
        "disposition": disposition,
        "next_gate": next_gate,
        "source_verifier": SOURCE_VERIFIER,
        "source_verifier_notebook_sha256": SOURCE_VERIFIER_NOTEBOOK_SHA256,
        "source_evidence_zip_sha256": SOURCE_EVIDENCE_ZIP_SHA256,
        "source_execution_log_sha256": SOURCE_EXECUTION_LOG_SHA256,
        "source_first_divergence": SOURCE_FIRST_DIVERGENCE,
        "source_failure_class": SOURCE_FAILURE_CLASS,
        "controlled_site_bootstrap_confirmed": controlled_confirmed,
        "package_installation_performed": False,
        "wheelhouse_attached": False,
        "network_access_requested": False,
        "model_loaded": False,
        "model_requests_performed": 0,
        "benchmark_trajectory_requests_performed": 0,
        "qualification_claimed": False,
        "credentials_used": False,
        "customer_data_used": False,
        "external_spend": 0,
    }

    for index, record in enumerate(records, start=1):
        role = str(record["command_role"])
        write_record(f"10_{index:02d}_{role}.json", record)
    write_record("90_summary.json", summary)

    payloads = tuple(sorted(EVIDENCE_ROOT.iterdir(), key=lambda item: item.name))
    manifest = {
        "schema_version": "1.0.0",
        "entries": [
            {
                "path": path.name,
                "sha256": streaming_sha256(path),
                "size_bytes": path.stat().st_size,
            }
            for path in payloads
        ],
    }
    write_record("99_evidence_sha256.json", manifest)

    with zipfile.ZipFile(
        OUTPUT_ZIP,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as archive:
        for path in sorted(EVIDENCE_ROOT.iterdir(), key=lambda item: item.name):
            archive.write(path, arcname=path.name)

    print(f"artifact={OUTPUT_ZIP}")
    print(f"size_bytes={OUTPUT_ZIP.stat().st_size}")
    print(f"sha256={streaming_sha256(OUTPUT_ZIP)}")
    print(f"inspection_status={summary['inspection_status']}")
    print(f"disposition={summary['disposition']}")
    print(
        "controlled_site_bootstrap_confirmed="
        + str(controlled_confirmed).lower()
    )
    print(f"next_gate={summary['next_gate']}")
    print("package_installation_performed=false")
    print("model_requests_performed=0")
    print("qualification_claimed=false")
    print("upload_only_this_file=true")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
